In [1]:
import pandas as pd
import numpy as np

# Define standard UCI Cleveland column names
columns = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 
    'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 
    'ca', 'thal', 'target'
]

# Load dataset and handle '?' missing values
df_heart = pd.read_csv(
    'data/processed.cleveland.data', 
    names=columns, 
    na_values='?'
)

# Impute missing values (ca and thal) with mode/median
df_heart['ca'] = df_heart['ca'].fillna(df_heart['ca'].mode()[0])
df_heart['thal'] = df_heart['thal'].fillna(df_heart['thal'].mode()[0])

# Convert multi-class target (0-4) to binary target (0 = No Disease, 1 = Heart Disease Risk)
df_heart['target'] = (df_heart['target'] > 0).astype(int)

print(f"Dataset shape: {df_heart.shape}")
print(df_heart.head())

Dataset shape: (303, 14)
    age  sex   cp  trestbps   chol  fbs  restecg  thalach  exang  oldpeak  \
0  63.0  1.0  1.0     145.0  233.0  1.0      2.0    150.0    0.0      2.3   
1  67.0  1.0  4.0     160.0  286.0  0.0      2.0    108.0    1.0      1.5   
2  67.0  1.0  4.0     120.0  229.0  0.0      2.0    129.0    1.0      2.6   
3  37.0  1.0  3.0     130.0  250.0  0.0      0.0    187.0    0.0      3.5   
4  41.0  0.0  2.0     130.0  204.0  0.0      2.0    172.0    0.0      1.4   

   slope   ca  thal  target  
0    3.0  0.0   6.0       0  
1    2.0  3.0   3.0       1  
2    2.0  2.0   7.0       1  
3    3.0  0.0   3.0       0  
4    1.0  0.0   3.0       0  


In [3]:
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

# 1. Separate features and target
X = df_heart.drop('target', axis=1)
y = df_heart['target']

# 2. Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Create ML pipeline (Scaling + Random Forest Classifier)
heart_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42))
])

# 4. Train model
heart_pipeline.fit(X_train, y_train)

# 5. Evaluate model
y_pred = heart_pipeline.predict(X_test)
print(f"Accuracy Score: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))

# 6. Save model to models directory
joblib.dump(heart_pipeline, 'models/heart_model.pkl')
print("✅ Saved heart_model.pkl successfully in models/ folder!")

Accuracy Score: 88.52%

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.85      0.89        33
           1       0.84      0.93      0.88        28

    accuracy                           0.89        61
   macro avg       0.89      0.89      0.89        61
weighted avg       0.89      0.89      0.89        61

✅ Saved heart_model.pkl successfully in models/ folder!


In [4]:
# Train Diabetes Risk Model
df_diabetes = pd.read_csv('data/diabetes.csv')

# Handle zero values in clinical columns (common in Pima dataset)
zero_cols = ['Glucose', 'BloodPressure', 'BMI', 'Insulin']
df_diabetes[zero_cols] = df_diabetes[zero_cols].replace(0, np.nan)
df_diabetes.fillna(df_diabetes.median(), inplace=True)

# Separate features and target
X_d = df_diabetes.drop('Outcome', axis=1)
y_d = df_diabetes['Outcome']

# Split data
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_d, y_d, test_size=0.2, random_state=42, stratify=y_d
)

# Pipeline & Training
diabetes_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42))
])

diabetes_pipeline.fit(X_train_d, y_train_d)

# Evaluate & Save
y_pred_d = diabetes_pipeline.predict(X_test_d)
print(f"Diabetes Model Accuracy: {accuracy_score(y_test_d, y_pred_d) * 100:.2f}%\n")
print(classification_report(y_test_d, y_pred_d))

joblib.dump(diabetes_pipeline, 'models/diabetes_model.pkl')
print("✅ Saved diabetes_model.pkl successfully in models/ folder!")

Diabetes Model Accuracy: 73.38%

              precision    recall  f1-score   support

           0       0.77      0.84      0.80       100
           1       0.64      0.54      0.59        54

    accuracy                           0.73       154
   macro avg       0.71      0.69      0.69       154
weighted avg       0.73      0.73      0.73       154

✅ Saved diabetes_model.pkl successfully in models/ folder!
